## Research question: Is algebra a thing in brain space?

Single concept perturbation


In [ ]:
import pickle
import sys
sys.path.append("../")
sys.path.append("../../")

from hparams import HParams
from hps import Hyperparams
from vae import VAE

from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.manifold import TSNE
import seaborn as sns
import nltk
from nltk.corpus import stopwords
import string
import numpy as np
import os
import glob
from os.path import join as opj
import h5py  
import matplotlib.pyplot as plt
import pandas as pd
import nibabel as nib
from scipy.io import loadmat
import torch
from torch.utils.data import Dataset, Subset, DataLoader
import json
from PIL import Image
# from diffusers import VersatileDiffusionPipeline
# from diffusers import VersatileDiffusionDualGuidedPipeline
from diffusers.models import AutoencoderKL, DualTransformer2DModel, Transformer2DModel, UNet2DConditionModel
from versatile_diffusion_dual_guided import VersatileDiffusionDualGuidedPipeline
from versatile_diffusion_dual_guided_fake_images import VersatileDiffusionDualGuidedFromCLIPEmbeddingPipeline
from autoencoder import *
from torchsummary import summary
import torchvision
import tqdm
from sklearn.linear_model import Ridge
import pickle
import wandb
from diffusers.utils import (
    PIL_INTERPOLATION,
    deprecate,
    is_accelerate_available,
    is_accelerate_version,
    logging,
    randn_tensor,
    replace_example_docstring,
)

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

from encoding_models import *
from decoding import *
import nilearn
from scipy import stats
from nilearn import plotting
import matplotlib.patches as patches
from torch.nn import functional as F

In [ ]:
def wrap_text(input_string):
    # Split the input string into words
    words = input_string.split()

    # Initialize an empty list to store the modified words
    modified_words = []

    # Iterate through the words and add a newline after every two words
    for i in range(0, len(words), 2):
        two_words = ' '.join(words[i:i+2])
        modified_words.append(two_words)

    # Join the modified words with newline characters
    result = '\n'.join(modified_words)

    return result

In [ ]:
base_path="/home/matteo/brain-diffuser/data"
timeseries_path=opj(base_path,"nsddata_timeseries")
betas_path=opj(base_path,"nsddata_betas")

stimuli_path=opj(base_path,"nsddata_stimuli","stimuli","nsd")
stim_file_path=opj(stimuli_path,"nsd_stimuli.hdf5")
sub="subj05"
mod="func1pt8mm"
subj_data_path=opj(timeseries_path,"ppdata",sub,mod,"timeseries")
subj_betas_path=opj(betas_path,"ppdata",sub,mod,"betas_assumehrf")

subj_betas_roi_extracted_path=opj(base_path,"processed_roi",sub,mod)

stim_order_path=opj(base_path,"nsddata","experiments","nsd","nsd_expdesign.mat")
stim_info_path=opj(base_path,"nsddata","experiments","nsd","nsd_stim_info_merged.csv")
stim_captions_train_path=opj(base_path,"nsddata_stimuli","stimuli","nsd","annotations",f"captions_train2017.json")
stim_captions_val_path=opj(base_path,"nsddata_stimuli","stimuli","nsd","annotations",f"captions_val2017.json")

processed_data=opj(base_path,"processed_data",sub)
sub_idx=int(sub.split("0")[-1])



In [ ]:
fmri_train_data=opj(processed_data,f"nsd_train_fmriavg_nsdgeneral_sub{sub_idx}.npy")


In [ ]:
fmri_test_data=opj(processed_data,f"nsd_test_fmriavg_nsdgeneral_sub{sub_idx}.npy")
imgs_test_data=opj(processed_data,f"nsd_test_stim_sub{sub_idx}.npy")
captions_test_data=opj(processed_data, f"nsd_test_cap_sub{sub_idx}.npy")

        
fmri_test_data=np.load(fmri_test_data)
imgs_test_data=np.load(imgs_test_data).astype(np.uint8)
captions_test_data=np.load(captions_test_data,allow_pickle=True)

In [ ]:
# load normalizing values 
mean=torch.load(f"../../models/{sub}/train_fmri_mean.pt").numpy()
std=torch.load(f"../../models/{sub}/train_fmri_std.pt").numpy()


fmri_test_data= (fmri_test_data-mean)/std
# fmri_test_data=np.nan_to_num(fmri_test_data)


In [ ]:
device="cuda:1"

In [ ]:
data_path="/home/matteo/data/CelebA"
img_path=opj(data_path,"Img","img_align_celeba")
attr_path=opj(data_path,"Anno","list_attr_celeba.csv")

## Load CLIP

In [ ]:
from PIL import Image
import requests
from transformers import AutoProcessor, CLIPModel

discriminator = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(
    text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True
)
inputs = {k:v.to(device) for k,v in inputs.items()}
# outputs = discriminator(**inputs)
# logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
# probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities

In [ ]:
train_imgs=np.load(opj(processed_data,f"nsd_train_stim_sub{sub_idx}.npy")).astype(np.uint8)
train_fmri= np.load(fmri_train_data)




In [ ]:
train_fmri = (train_fmri-mean)/std

In [ ]:
## encode all the images in batch with clip
BS=128

img_embeds=[]
with torch.no_grad():
    for batch_idx in tqdm.trange(0,len(train_imgs)//BS+1,1):
        batch = train_imgs[batch_idx*BS:(batch_idx+1)*BS]
        batch = [Image.fromarray(i) for i in batch]
        
        # apply processor
        inputs = processor(images=batch, return_tensors="pt", padding=True)
        inputs = {k:v.to(device) for k,v in inputs.items()}
        emb = discriminator.get_image_features(inputs["pixel_values"]).cpu()
        img_embeds.append(emb)
        
img_embeds = torch.cat(img_embeds,0)
        

## Decoder

In [ ]:
subj_path=opj("/home/matteo/explore_NSD","models",sub)
brain_decoder=BrainDiffuserPretrainedDecoder(subj_path=subj_path,device=device)

## Normalization

After the gaussian assumption

$\sigma^2_z$ is the variance of the original image that is already standardized

We encode two concepts
$\sigma^2_c=\sigma^2_1 +\sigma^2_2 \sim 2\sigma^2$

Then we scale them by $\alpha$ and add this difference to the unperturbed fMRI

The final scaling factor should be $\sqrt{(1+2\alpha^2)}$

In [ ]:
# np.sqrt(1+2*1**2),np.sqrt(1+2*2**2), np.sqrt(1+2*3**2), np.sqrt(1+2*4**2)
# # 

In [ ]:
def run_algebra_experiment_on_brain_space_single_pert(fmri_inputs=None, kind="text", pert="",fmri_pert=None, fmri_stim=None,thr_perc=None,l2_norm=True, N=100):
    
    ## look for concepts fMRI
#     inputs = processor(text=[positive, negative],return_tensors="pt", padding=True)
#     txt_embeds=discriminator.get_text_features(inputs["input_ids"].to(device)).cpu()
    
#     probs = (img_embeds@txt_embeds.T).softmax(1)
#     pos_indices = np.where(probs[:,0]>0.95)[0]
#     neg_indices = np.where(probs[:,1]>0.95)[0]
    
#     fmri_positive=train_fmri[pos_indices][:N].mean(0)
#     fmri_negative=train_fmri[neg_indices][:N].mean(0)
    # fmri_positive = positive
    # fmri_negative = negative
    
    fmri_encoded=torch.tensor(fmri_inputs).unsqueeze(0)
    
    
    
    ## what if they have different shapes?
        
    # plain_decoded,_=brain_decoder.decode(fmri_encoded,num_inference_steps=37)   

    fmri_difference = torch.tensor(fmri_pert)
    
    n4=5.7445
    n3=4.3588
    n2=3.
    n1=1.7320
    
    c1=0.25
    c2=0.5
    c3=0.75
    c4=1
    
    if thr_perc is not None:
        print(f"thresholding concept at {thr_perc}")
        #trheshold the difference vector in brain space
        if thr_perc>1:
        
            thr=torch.quantile(fmri_difference.abs(),thr_perc/100)
        else:
            thr=torch.quantile(fmri_difference.abs(),thr_perc)

        mask = (fmri_difference.abs() <= thr)

        # Set values below the threshold to zero
        fmri_difference[mask] = 0
    
        n1=1
        n2=1
        n3=1
        n4=1
        
        c1=1
        c2=2
        c3=3
        c4=4
    
    fmri_algebra=torch.cat([(fmri_encoded-c4*fmri_difference)/n4,
                            (fmri_encoded-c3*fmri_difference)/n3, 
                              (fmri_encoded-c2*fmri_difference)/n2,
                              (fmri_encoded-c1*fmri_difference)/n1,
                              fmri_encoded,
                              (fmri_encoded+c1*fmri_difference)/n1,
                              (fmri_encoded+c2*fmri_difference)/n2,
                              (fmri_encoded+c3*fmri_difference)/n3,
                              (fmri_encoded+c4*fmri_difference)/n4
                           ],0)
    
    
    if l2_norm:
        norm=fmri_encoded.norm(dim=1) #norm before
        fmri_algebra=F.normalize(fmri_algebra,dim=1)
        #reset the previous norm
        fmri_algebra=norm.unsqueeze(1)*fmri_algebra
        
    # print(fmri_algebra.shape)
    
    algebra_imgs,_=brain_decoder.decode(fmri_algebra,num_inference_steps=37)
    

    if kind=="fmri":
        fig, axs = plt.subplots(1,10, figsize=(30,6),subplot_kw={'aspect': 1})

        # titles=[f"Original {i}({positive}-{negative})" for i in ["-3","-2","-1","+1","+2","+3"]]
        
        axs[0].imshow(fmri_stim)
        axs[0].axis("off")
        axs[0].set_title("Stimulus shown")


#         axs[1].imshow(plain_decoded[0])
#         axs[1].axis("off")
#         axs[1].set_title("Original decoded")

        titles=[f"Original -{i}({pert})" for i in [c1,c2,c3,c4][::-1]]+["Original decoded"]+[f"Original +{i}({pert})" for i in  [c1,c2,c3,c4]]
        for i in range(9):
            axs[i+1].imshow(algebra_imgs[i])
            axs[i+1].axis("off")
            axs[i+1].set_title(wrap_text(titles[i]))
            
            
        # Add a border around the image
        border_color = 'black'  # You can specify the border color
        border_width = 3      # You can specify the border width in pixels

        # Get the dimensions of the image
        height, width, _ = np.array(algebra_imgs[0]).shape

        # Create a rectangle patch and add it to the axes
        border_rect = patches.Rectangle((0, 0), width, height, linewidth=border_width, edgecolor=border_color, fill=False)
        axs[5].add_patch(border_rect)
        
    
    plain_decoded=algebra_imgs[4]
    #save difference
    
    return plain_decoded, algebra_imgs, fmri_difference,fig
    
    

In [ ]:
# F.normalize(a.unsqueeze(0))

## ALGEBRA IN BRAIN SPACE

In [ ]:
thr_outputs=[]

In [ ]:
## Define all experiments

modality="fmri"
N=100
thr_outputs=[]

## Remove useful indices 
indices=[25,31,68,121,126,318,384,492,531,606,702,860]

indices2=[70,116,165,261,278,363,451,774]
indices3=[41,205,230,411,428,446,502,777]
extra=[95,905]
comparison_indices=indices+indices2+indices3+extra

indices = comparison_indices
indices=np.arange(75).tolist()   ##UNCOMMENT THESE 2 LINES TO OBTAIN 100 images instead of 30
indices = sorted(list(set(comparison_indices + indices)))

print(len(indices))

# indices = [12] #mini check con uno solo

prompts=fmri_test_data[indices]
stims=imgs_test_data[indices]

difference_pairs=[("man","woman"),("indoor scene","outdoor scene"),("day","night"),("summer","winter"),("sad","happy"),("cat","dog"), ("still", "moving"), ("left","right"), ("top","down"), ("crowded","empty")]


import os
import torch
from os.path import join as opj



# for thr in [0, 25, 50, 75, 90,95]:
for thr in [75,90,95]:
# for thr in [25,50,75,90,95]:
    print(f"INFO thr: {thr}")
    out_dir=f"models/{sub}/encoding/algebra_l2_SINGLE_thr_{thr}"
    os.makedirs(out_dir,exist_ok=True)
    ## Text Experiments




    # thr = 0.90


    # Assuming the following functions and variables are defined elsewhere in your code:
    # - prompts: list of prompts
    # - stims: list of stimuli
    # - difference_pairs: list of tuples containing the positive and negative pairs
    # - out_dir: output directory
    # - modality: modality type
    # - thr: threshold percentage
    # - run_algebra_experiment_on_brain_space: function to run the experiment

    # Initialize the dictionary to collect all outputs
    outputs = {}


        # Initialize the dictionary for the current prompt and stim

    for diff in difference_pairs:
        

        positive, negative = diff
        
        inputs = processor(text=[positive, negative],return_tensors="pt", padding=True)
        txt_embeds=discriminator.get_text_features(inputs["input_ids"].to(device)).cpu()
        probs = (img_embeds@txt_embeds.T).softmax(1)
        # pos_indices = np.where(probs[:,0]>0.95)[0]
        # neg_indices = np.where(probs[:,1]>0.95)[0]
        
        pos_indices = probs[:,0].argsort()[-N:].detach().numpy()
        neg_indices = probs[:,1].argsort()[-N:].detach().numpy()
        

        fmri_positive=train_fmri[pos_indices][:N].mean(0)
        fmri_negative=train_fmri[neg_indices][:N].mean(0)
        
        for pert,fmri_pert in zip(diff,[fmri_positive,fmri_negative]):
            print("[INFO] Running", pert)
            target_dir = opj(out_dir, modality, f"{pert}_thr_{thr}")
            os.makedirs(target_dir, exist_ok=True)
            outputs[f"{pert}"] = {}
            
                        # Plotting the grid of images
            if pert == positive:
                grid_images = train_imgs[pos_indices]
                title = f"Positive Images for {positive} (thr={thr})"
            else:
                grid_images = train_imgs[neg_indices]
                title = f"Negative Images for {negative} (thr={thr})"

            fig, axes = plt.subplots(10, 10, figsize=(20, 20))
            fig.suptitle(title, fontsize=20)
            for img, ax in zip(grid_images[:100], axes.flat):
                ax.imshow(img)  # Assuming images are in a format plt.imshow can display
                ax.axis('off')
            # plt.tight_layout()
            plt.savefig(opj(target_dir, f"{pert}_images_grid.png"))
            plt.show()



            for i, (prompt, stim) in enumerate(list(zip(prompts, stims))):


                plain_decoded, algebra_imgs, fmri_difference, fig = run_algebra_experiment_on_brain_space_single_pert(
                    fmri_inputs=prompt,
                    kind=modality,
                    pert=pert,
                    fmri_pert= fmri_pert,
                    fmri_stim=stim,
                    thr_perc=thr
                )

                # Save stuff
                torch.save(fmri_difference, opj(target_dir, f"diff_{i}.pt"))
                fig.savefig(opj(target_dir, f"algebra_{i}.png"))
                svg_filename = opj(target_dir, f"algebra_{i}.svg")
                fig.savefig(svg_filename, format='svg', dpi=300)

                # Store results in the dictionary
                outputs[f"{pert}"][i]= {
                    "fmri_difference": fmri_difference,
                    "algebra_imgs": algebra_imgs,
                    "plain_decoded": plain_decoded,
                    "figure": fig,
                    "pos_num": len(pos_indices),
                    "neg_num": len(neg_indices)
                }


            thr_outputs.append(outputs)
        
        
        
        

In [ ]:
print("ok")

In [ ]:
print("ok")

In [ ]:
# train_imgs

In [ ]:
# if pert == positive:
#     grid_images = train_imgs[pos_indices]
#     title = f"Positive Images for {positive} (thr={thr})"
# else:
#     grid_images = train_imgs[neg_indices]
#     title = f"Negative Images for {negative} (thr={thr})"

# fig, axes = plt.subplots(10, 10, figsize=(20, 20))
# fig.suptitle(title, fontsize=20)
# for img, ax in zip(grid_images[:100], axes.flat):
#     ax.imshow(img)  # Assuming images are in a format plt.imshow can display
#     ax.axis('off')
# # plt.tight_layout()
# # plt.savefig(opj(target_dir, f"{pert}_images_grid.png"))
# plt.show()